In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, yaml
import torch

Mounted at /content/drive


In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
print('CUDA disponible :', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 9.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
CUDA disponible : True


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake"

drive_training_data = f"{PROJECT_ROOT}/xView/training_data"
local_training_data = "/content/xView_local/training_data"

dota_root = f"{PROJECT_ROOT}/DOTA"
local_dota_yolo = "/content/dota_pour_yolo"

drive_v3_run = f"{PROJECT_ROOT}/xView/runs/xview_yolov8s_merged_v3"
local_v3_run = "/content/xView_local/runs/xview_yolov8s_merged_v3"

run_name = "xview_dota_merged_v1"
drive_finetune_runs = f"{PROJECT_ROOT}/xView/runs"
local_finetune_run = f"/content/xView_local/runs/{run_name}"

In [ ]:
import subprocess

zip_dota_path = f"{PROJECT_ROOT}/DOTA/dota_train_val.zip"

if not os.path.exists(zip_dota_path):
    cmd = (
        f'cd "{dota_root}" && '
        f'zip -rq "{zip_dota_path}" '
        f'train/images train/labels_remapped '
        f'val/images val/labels_remapped'
    )
    subprocess.run(cmd, shell=True)
    print("Archive créée sur Drive.")
else:
    print("Archive déjà existante sur Drive.")

local_zip_path = "/content/dota_train_val.zip"

if not os.path.exists(local_zip_path):
    shutil.copy(zip_dota_path, local_zip_path)
    print("Zip copié en local.")

if not os.path.exists(local_dota_yolo):
    subprocess.run(f'unzip -q "{local_zip_path}" -d "{local_dota_yolo}"', shell=True)
    print("Archive décompressée en local.")

for split in ['train', 'val']:
    n_img = len(os.listdir(f"{local_dota_yolo}/{split}/images"))
    print(f"{split} : {n_img} images en local")

Archive créée sur Drive.
Zip copié en local.


KeyboardInterrupt: 

In [ ]:
import subprocess

def copier_avec_rsync(src, dst):
    os.makedirs(dst, exist_ok=True)
    cmd = f'rsync -a --info=progress2 "{src}/" "{dst}/"'
    subprocess.run(cmd, shell=True)

print("--- Copie images xView ---")
copier_avec_rsync(f"{drive_training_data}/images", f"{local_training_data}/images")

print("--- Copie labels_merged xView ---")
copier_avec_rsync(f"{drive_training_data}/labels_merged", f"{local_training_data}/labels")

n_img = sum(len(os.listdir(f"{local_training_data}/images/{s}")) for s in ['train', 'val'])
n_lbl = sum(len(os.listdir(f"{local_training_data}/labels/{s}")) for s in ['train', 'val'])
print(f"xView : {n_img} images, {n_lbl} labels en local")

with open(f"{drive_training_data}/data_merged.yaml") as f:
    data_yaml_merged = yaml.safe_load(f)
xview_names = data_yaml_merged["names"]
print("nc xView :", len(xview_names))

--- Copie images xView ---
--- Copie labels_merged xView ---
xView : 12640 images, 12640 labels en local
nc xView : 29


In [ ]:
nouvelles_classes = ['Harbor', 'Ground Track Field', 'Soccer Ball Field', 'Tennis Court',
                      'Swimming Pool', 'Baseball Diamond', 'Roundabout',
                      'Basketball Court', 'Bridge']
noms_finaux = xview_names + nouvelles_classes
print(f"Taxonomie finale : {len(noms_finaux)} classes")

for split in ['train', 'val']:
    d = os.path.join(dota_root, split, 'labels_remapped')
    print(f"{split} : {len(os.listdir(d))} labels remappés trouvés sur Drive")

Taxonomie finale : 38 classes
train : 15749 labels remappés trouvés sur Drive
val : 5297 labels remappés trouvés sur Drive


In [ ]:
import subprocess

def copier_avec_rsync(src, dst):
    os.makedirs(dst, exist_ok=True)
    cmd = f'rsync -a --info=progress2 "{src}/" "{dst}/"'
    subprocess.run(cmd, shell=True)

for split in ['train', 'val']:
    print(f"--- Copie {split}/images ---")
    copier_avec_rsync(f"{dota_root}/{split}/images", f"{local_dota_yolo}/{split}/images")
    print(f"--- Copie {split}/labels_remapped ---")
    copier_avec_rsync(f"{dota_root}/{split}/labels_remapped", f"{local_dota_yolo}/{split}/labels")

for split in ['train', 'val']:
    n_img = len(os.listdir(f"{local_dota_yolo}/{split}/images"))
    n_lbl = len(os.listdir(f"{local_dota_yolo}/{split}/labels"))
    print(f"{split} : {n_img} images, {n_lbl} labels en local")

--- Copie train/images ---


KeyboardInterrupt: 

In [ ]:
data_yaml_fusionne = {
    "path": "",
    "train": [f"{local_training_data}/images/train", f"{local_dota_yolo}/train/images"],
    "val": [f"{local_training_data}/images/val", f"{local_dota_yolo}/val/images"],
    "nc": len(noms_finaux),
    "names": noms_finaux
}

chemin_yaml_fusion = "/content/data_xview_dota_merged.yaml"
with open(chemin_yaml_fusion, "w") as f:
    yaml.dump(data_yaml_fusionne, f, sort_keys=False)

print("nc :", data_yaml_fusionne["nc"])

In [ ]:
def backup_to_drive(trainer):
    run_dir = str(trainer.save_dir)
    dest = f"{drive_finetune_runs}/{os.path.basename(run_dir)}"
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f"[backup] epoch {trainer.epoch} sauvegardé sur Drive → {dest}")

In [ ]:
drive_run_path = f"{drive_finetune_runs}/{run_name}"
last_ckpt_drive = f"{drive_run_path}/weights/last.pt"

if os.path.exists(last_ckpt_drive):
    print("Run existant détecté sur Drive, reprise de l'entraînement en cours...")
    os.makedirs(os.path.dirname(local_finetune_run), exist_ok=True)
    if not os.path.exists(local_finetune_run):
        shutil.copytree(drive_run_path, local_finetune_run)
    model = YOLO(f"{local_finetune_run}/weights/last.pt")
    model.add_callback("on_train_epoch_end", backup_to_drive)
    results = model.train(resume=True)
else:
    print("Aucun run existant, démarrage du fine-tuning depuis best.pt v3...")
    if not os.path.exists(local_v3_run):
        shutil.copytree(drive_v3_run, local_v3_run)
    model = YOLO(f"{local_v3_run}/weights/best.pt")
    model.add_callback("on_train_epoch_end", backup_to_drive)
    results = model.train(
        data=chemin_yaml_fusion,
        epochs=150,
        imgsz=640,
        batch=16,
        patience=25,
        save_period=1,
        cache='disk',
        project="/content/xView_local/runs",
        name=run_name
    )

In [ ]:
import subprocess

chemin_zip = "/content/dota_train_val.zip"
taille_go = os.path.getsize(chemin_zip) / (1024*1024*1024)
print(f"Taille du zip : {taille_go:.2f} Go")

dossier_morceaux = "/content/dota_zip_parts"
os.makedirs(dossier_morceaux, exist_ok=True)

subprocess.run(
    f'split -b 1500M "{chemin_zip}" "{dossier_morceaux}/dota_part_"',
    shell=True
)

morceaux = sorted(os.listdir(dossier_morceaux))
print(f"{len(morceaux)} morceaux créés :", morceaux)

Taille du zip : 24.81 Go
17 morceaux créés : ['dota_part_aa', 'dota_part_ab', 'dota_part_ac', 'dota_part_ad', 'dota_part_ae', 'dota_part_af', 'dota_part_ag', 'dota_part_ah', 'dota_part_ai', 'dota_part_aj', 'dota_part_ak', 'dota_part_al', 'dota_part_am', 'dota_part_an', 'dota_part_ao', 'dota_part_ap', 'dota_part_aq']


In [ ]:
for m in morceaux:
    files.download(os.path.join(dossier_morceaux, m))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil

dossier_a_supprimer = "/content/dota_zip_parts"

if os.path.exists(dossier_a_supprimer):
    shutil.rmtree(dossier_a_supprimer)
    print(f"{dossier_a_supprimer} supprimé.")
else:
    print("Le dossier n'existe pas (déjà supprimé ou jamais créé).")

/content/dota_zip_parts supprimé.
